# Safety events Databricks upload

Load the `Data` worksheet from `data/NearMissCompilation.xlsx`, remove rows with missing Month/Year, rename Month/Year to Date, derive numeric Near Miss/pSIF/SIF flags, and map division labels. All source columns are retained apart from the date-column rename. The destination is `integrated_operations_dev.analytics.safety_events`.


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

FILE_SAFETY = "data/NearMissCompilation.xlsx"
SHEET_NAME = "Data"


In [ ]:
if not Path(FILE_SAFETY).is_file():
    raise FileNotFoundError(f"Workbook not found: {FILE_SAFETY}")

safety = pd.read_excel(FILE_SAFETY, sheet_name=SHEET_NAME)


In [ ]:
REQUIRED_COLUMNS = [
    "Month/Year", "Division", "Site", "Code", "Category",
    "Near Miss (Y/N)", "pSIF (Y/N)", "SIF (Y/N)",
]
DIVISION_MAP = {
    "CWI": "DS C2 & Weapons Integration",
    "WS": "DS Weapon Systems",
    "SDS": "DS Strategic Deterrent Systems",
    "HQ": "DS Sector HQ",
}
FLAG_SOURCES = {
    "Near Miss": "Near Miss (Y/N)",
    "pSIF": "pSIF (Y/N)",
    "SIF": "SIF (Y/N)",
}


def prepare_safety(safety):
    missing = [column for column in REQUIRED_COLUMNS if column not in safety.columns]
    if missing:
        raise ValueError(f"Missing safety columns: {missing}")
    if "Date" in safety.columns:
        raise ValueError("The source contains both Month/Year and Date columns.")

    processed = safety.copy(deep=True)
    processed.dropna(axis=0, subset=["Month/Year"], inplace=True)
    processed.rename(columns={"Month/Year": "Date"}, inplace=True)
    processed["Date"] = pd.to_datetime(processed["Date"], errors="raise")
    if processed.empty:
        raise ValueError("No dated safety rows available for upload.")
    if processed["Date"].isna().any():
        raise ValueError("Safety dates must be populated.")

    # Unrecognized or blank flags remain null, rather than being counted as zero.
    for column, source in FLAG_SOURCES.items():
        processed[column] = processed[source].map({"Y": 1, "N": 0}).astype("Int64")
    processed["Division"] = processed["Division"].replace(DIVISION_MAP)
    return processed


processed = prepare_safety(safety)
print(f"Source rows: {len(safety):,}")
print(f"Dropped rows without Month/Year: {len(safety) - len(processed):,}")
print(f"Upload-ready records: {len(processed):,}")
display(processed[["Date", "Division", "Site", "Code", "Category", "Near Miss", "pSIF", "SIF"]].head())
display(processed.groupby("Division", dropna=False).size().rename("Rows").reset_index())


## Upload to Databricks

Install dependencies in the notebook environment if needed:

```python
%pip install pandas openpyxl python-dotenv "databricks-sql-connector>=3.0.0"
```

The final cell uses the same local, untracked `upload.env` as the OTD uploader: `DATABRICKS_HOST`, `DATABRICKS_TOKEN`, and `DATABRICKS_SQL_WAREHOUSE_ID`. The warehouse value is the ID after `/sql/1.0/warehouses/`, rather than the full HTTP path.

Set `RUN_DATABRICKS_UPLOAD = True` to replace `integrated_operations_dev.analytics.safety_events` with the complete prepared dataset. The upload follows the OTD process: load a unique staging table in the same schema, verify its row count, publish with one Delta replacement statement, and remove staging afterward. Loading or validation failures before publication preserve the existing target. Column mapping retains workbook headers containing spaces and parentheses. Dates are stored as timestamps and derived flags as nullable integers; other column types follow the workbook data. Schema creation and access are managed by the platform.


In [ ]:
# Replace the dev safety table only when explicitly enabled.
RUN_DATABRICKS_UPLOAD = False

if RUN_DATABRICKS_UPLOAD:
    import math
    import re
    import warnings
    from uuid import uuid4
    from urllib.parse import urlparse
    from dotenv import dotenv_values
    from databricks import sql

    env_path = Path("upload.env")
    if not env_path.is_file():
        raise FileNotFoundError(f"Missing Databricks connection file: {env_path}")
    credentials = dotenv_values(env_path)
    required = ["DATABRICKS_HOST", "DATABRICKS_TOKEN", "DATABRICKS_SQL_WAREHOUSE_ID"]
    settings = {name: (credentials.get(name) or "").strip() for name in required}
    missing = [name for name, value in settings.items() if not value]
    if missing:
        raise ValueError(f"Missing upload.env settings: {missing}")

    host = settings["DATABRICKS_HOST"]
    parsed_host = urlparse(host if "://" in host else f"https://{host}")
    if (parsed_host.scheme != "https" or not parsed_host.hostname
            or parsed_host.username or parsed_host.password
            or parsed_host.path not in ("", "/") or parsed_host.query
            or parsed_host.fragment or parsed_host.port):
        raise ValueError("DATABRICKS_HOST must be a Databricks workspace hostname or HTTPS origin.")
    warehouse_id = settings["DATABRICKS_SQL_WAREHOUSE_ID"]
    if not re.fullmatch(r"[A-Za-z0-9_-]+", warehouse_id):
        raise ValueError("DATABRICKS_SQL_WAREHOUSE_ID must be the warehouse ID, not its HTTP path.")

    # Keep the destination fixed to dev, independent of SQL Server settings in upload.env.
    catalog, schema, table = "integrated_operations_dev", "analytics", "safety_events"
    def quote_identifier(value):
        return "`" + str(value).replace("`", "``") + "`"

    target = ".".join(map(quote_identifier, [catalog, schema, table]))
    staging = ".".join(map(quote_identifier, [catalog, schema, f"safety_events_upload_{uuid4().hex}"]))
    UPLOAD_COLUMNS = list(processed.columns)
    flag_columns = {"Near Miss", "pSIF", "SIF"}
    column_types = {}
    for column in UPLOAD_COLUMNS:
        dtype = processed[column].dtype
        if column == "Date" or pd.api.types.is_datetime64_any_dtype(dtype):
            column_types[column] = "TIMESTAMP"
        elif column in flag_columns or pd.api.types.is_integer_dtype(dtype):
            column_types[column] = "BIGINT"
        elif pd.api.types.is_bool_dtype(dtype):
            column_types[column] = "BOOLEAN"
        elif pd.api.types.is_numeric_dtype(dtype):
            column_types[column] = "DOUBLE"
        else:
            column_types[column] = "STRING"
    column_sql = ", ".join(map(quote_identifier, UPLOAD_COLUMNS))
    definition = ", ".join(f"{quote_identifier(c)} {column_types[c]}" for c in UPLOAD_COLUMNS)

    # Convert pandas scalars and missing values before making any database changes.
    def upload_value(column, value):
        if pd.isna(value):
            return None
        data_type = column_types[column]
        if data_type == "TIMESTAMP":
            return pd.Timestamp(value).to_pydatetime()
        if data_type == "BIGINT":
            return int(value)
        if data_type == "BOOLEAN":
            return bool(value)
        if data_type == "DOUBLE":
            number = float(value)
            if not math.isfinite(number):
                raise ValueError(f"Non-finite safety value in {column}.")
            return number
        return str(value)

    rows = [
        tuple(upload_value(column, value) for column, value in zip(UPLOAD_COLUMNS, row))
        for row in processed[UPLOAD_COLUMNS].itertuples(index=False, name=None)
    ]
    if not rows:
        raise ValueError("No rows available for upload.")
    with sql.connect(
        server_hostname=parsed_host.hostname,
        http_path=f"/sql/1.0/warehouses/{warehouse_id}",
        access_token=settings["DATABRICKS_TOKEN"],
    ) as connection:
        with connection.cursor() as cursor:
            # Confirm schema access before loading. Schemas are managed by the platform.
            cursor.execute(f"DESCRIBE SCHEMA {quote_identifier(catalog)}.{quote_identifier(schema)}")
            cursor.fetchall()
            stage_created = False
            try:
                # Column mapping retains workbook headers, including spaces and parentheses.
                cursor.execute(
                    f"CREATE TABLE {staging} ({definition}) USING DELTA "
                    "TBLPROPERTIES ('delta.columnMapping.mode' = 'name')"
                )
                stage_created = True
                batch_size = 100
                row_placeholders = "(" + ", ".join("?" for _ in UPLOAD_COLUMNS) + ")"
                for offset in range(0, len(rows), batch_size):
                    batch = rows[offset:offset + batch_size]
                    placeholders = ", ".join(row_placeholders for _ in batch)
                    parameters = [value for row in batch for value in row]
                    cursor.execute(
                        f"INSERT INTO {staging} ({column_sql}) VALUES {placeholders}",
                        parameters,
                    )
                    print(f"Loaded {min(offset + batch_size, len(rows)):,} / {len(rows):,} rows.")

                cursor.execute(f"SELECT COUNT(*) FROM {staging}")
                staged_count = int(cursor.fetchone()[0])
                if staged_count != len(rows):
                    raise RuntimeError(f"Staging row-count mismatch: {staged_count} != {len(rows)}")

                # Publish the complete dataset with one Delta replacement statement.
                cursor.execute(
                    f"CREATE OR REPLACE TABLE {target} USING DELTA "
                    "TBLPROPERTIES ('delta.columnMapping.mode' = 'name') "
                    f"AS SELECT {column_sql} FROM {staging}"
                )
                cursor.execute(f"SELECT COUNT(*) FROM {target}")
                after = int(cursor.fetchone()[0])
                if after != len(rows):
                    raise RuntimeError(f"Published row-count mismatch: {after} != {len(rows)}")
                print(f"Updated {catalog}.{schema}.{table}: {after:,} rows.")
            finally:
                if stage_created:
                    try:
                        cursor.execute(f"DROP TABLE IF EXISTS {staging}")
                    except Exception:
                        warnings.warn(f"Could not remove staging table {staging}; remove it when the connection is available.")
else:
    print("Databricks upload disabled. Set RUN_DATABRICKS_UPLOAD = True to replace integrated_operations_dev.analytics.safety_events.")
